# Compute flux for a given target

- For given energy E : plot count map for verification
- Model noise morphology and spectrum
- Model target ; Dirac delta ? Power Law ? 
- Include IRFs (exposure, PSF)
- Run either MLE or MCMC

### Setup

In [ ]:
import ROOT, glob, os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

In [ ]:
dirDir= "/home/users/l/lebrun/"
resultPath = dirDir + "results/"
figPath = dirDir + "figures/"
dataPath = dirDir + "data/"
dataFile = dataPath + "DmlNtup_Data_9years_FullPhotonData_AllSky.root"
exposurePath = "/dpnc/beegfs/dampe/tryf/ForKimberly/filtered_cos_theta_bins/exposureMaps/"
resPath = dirDir + "results/"
# exponent = 0


In [ ]:
import os
target = "Geminga" # Change this for output filenames

try:
    os.mkdir(figPath+target)
except FileExistsError:
    pass

figPath = figPath + target + "/"

In [ ]:
import os
import re
from astropy.coordinates import SkyCoord
from astropy import units as u

def parse_par_file(filename):
    """
    Parse a .par file and extract RAJ and DECJ values.
    
    Parameters:
    filename (str): Path to the .par file
    
    Returns:
    tuple: (raj_value, decj_value) or (None, None) if not found
    """
    raj_value = None
    decj_value = None
    
    try:
        with open(filename, 'r') as f:
            for line in f:
                line = line.strip()
                
                # Skip empty lines and comments
                if not line or line.startswith('#'):
                    continue
                
                # Split line into components
                parts = line.split()
                if len(parts) < 2:
                    continue
                
                parameter = parts[0].upper()
                
                # Extract RAJ value
                if parameter == 'RAJ':
                    # RAJ value is typically in format HH:MM:SS.ssss
                    raj_value = parts[1]
                
                # Extract DECJ value  
                elif parameter == 'DECJ':
                    # DECJ value is typically in format +/-DD:MM:SS.ssss
                    decj_value = parts[1]
    
    except FileNotFoundError:
        print(f"Error: Could not find file {filename}")
        return None, None
    except Exception as e:
        print(f"Error reading file {filename}: {e}")
        return None, None
    
    return raj_value, decj_value

def convert_ra_to_decimal(ra_str):
    """
    Convert RA string from HH:MM:SS.ssss format to decimal hours.
    
    Parameters:
    ra_str (str): RA in HH:MM:SS.ssss format
    
    Returns:
    float: RA in decimal hours
    """
    if ra_str is None:
        return None
    
    try:
        parts = ra_str.split(':')
        hours = float(parts[0])
        minutes = float(parts[1]) if len(parts) > 1 else 0
        seconds = float(parts[2]) if len(parts) > 2 else 0
        
        decimal_hours = hours + minutes/60.0 + seconds/3600.0
        return decimal_hours
    except (ValueError, IndexError):
        print(f"Error parsing RA: {ra_str}")
        return None

def convert_dec_to_decimal(dec_str):
    """
    Convert DEC string from +/-DD:MM:SS.ssss format to decimal degrees.
    
    Parameters:
    dec_str (str): DEC in +/-DD:MM:SS.ssss format
    
    Returns:
    float: DEC in decimal degrees
    """
    if dec_str is None:
        return None
    
    try:
        # Handle sign
        sign = 1
        if dec_str.startswith('+'):
            dec_str = dec_str[1:]
        elif dec_str.startswith('-'):
            sign = -1
            dec_str = dec_str[1:]
        
        parts = dec_str.split(':')
        degrees = float(parts[0])
        minutes = float(parts[1]) if len(parts) > 1 else 0
        seconds = float(parts[2]) if len(parts) > 2 else 0
        
        decimal_degrees = degrees + minutes/60.0 + seconds/3600.0
        return sign * decimal_degrees
    except (ValueError, IndexError):
        print(f"Error parsing DEC: {dec_str}")
        return None

# Main extraction code
parDirName = dirDir + f"data/{target}/"

# HERE you can enter your custom path to a .par file. Otherwise the script will look
# into data/{target} for a .par file
filename = None 

if filename is None:
    try:
        for file in os.listdir(parDirName[:-1]):
            if file.endswith(".par"):
                filename = os.path.join(parDirName, file)
                break
    except FileNotFoundError as e:
        print(f"Error: no par file found in subdirectory {parDirName}. Target: {target}")
        raise e

if filename is None:
    raise FileNotFoundError(f"No .par file found in {parDirName}")

# Extract RAJ and DECJ from the par file
raj_str, decj_str = parse_par_file(filename)

if raj_str is None or decj_str is None:
    print("Error: Could not find RAJ and/or DECJ in the .par file")
    print(f"RAJ: {raj_str}, DECJ: {decj_str}")
else:
    # Convert to decimal values
    ra_target = convert_ra_to_decimal(raj_str)
    dec_target = convert_dec_to_decimal(decj_str)
    
    print(f"RAJ: {ra_target} hours, DECJ: {dec_target} degrees")
    print(f"Original format - RAJ: {raj_str}, DECJ: {decj_str}")
    
    # Create SkyCoord object (same as original)
    if ra_target is not None and dec_target is not None:
        c = SkyCoord(ra=ra_target*u.hourangle, dec=dec_target*u.degree, frame='icrs')
        c = c.transform_to('galactic')
        print(f"Galactic coordinates: l={c.l}, b={c.b}")

### Energy binning

In [ ]:
Energy = 2.81 # GeV # TODO : assess whether we are doing only 1 energy bin or all in this notebook
Emin = 1
Emax_HET = 1e4
n_E_bins_EffArea = 40
n_E_bins_PSF = 8

E_bins_edge_EffArea = np.logspace(np.log10(Emin), np.log10(Emax_HET), n_E_bins_EffArea + 1)
E_bins_center_EffArea = np.sqrt(E_bins_edge_EffArea[:-1] * E_bins_edge_EffArea[1:])

E_bins_edge_PSF = np.logspace(np.log10(Emin), np.log10(Emax_HET), n_E_bins_PSF + 1)
E_bins_center_PSF = np.sqrt(E_bins_edge_PSF[:-1] * E_bins_edge_PSF[1:])

In [ ]:
def find_bin_indices(energy) :
    bin_idx_EffArea = np.searchsorted(E_bins_edge_EffArea, energy, side='right') -1
    bin_idx_PSF = np.searchsorted(E_bins_edge_PSF, energy, side='right') -1

    return bin_idx_EffArea, bin_idx_PSF

def find_bin_edges(energy):

    bin_idx_EffArea, bin_idx_PSF = find_bin_indices(energy)

    Ebin_lb_EffArea = E_bins_edge_EffArea[bin_idx_EffArea]
    Ebin_ub_EffArea = E_bins_edge_EffArea[bin_idx_EffArea + 1]
    Ebin_lb_PSF = E_bins_edge_PSF[bin_idx_PSF]
    Ebin_ub_PSF = E_bins_edge_PSF[bin_idx_PSF + 1]

    return Ebin_lb_EffArea, Ebin_ub_EffArea, Ebin_lb_PSF, Ebin_ub_PSF

def energy_str(energy) :
    bin_idx_EffArea, bin_idx_PSF = find_bin_indices(energy)

    energy_exposure_map = E_bins_center_EffArea[bin_idx_EffArea]
    energy_exposure_map_str = f"{energy_exposure_map:.3f}"

    energy_PSF = E_bins_center_PSF[bin_idx_PSF]
    energy_PSF_str = f"{energy_PSF:.3f}"

    return energy_exposure_map_str, energy_PSF_str

In [ ]:
# Find the bin index such that E_bins_edge_HET[bin_idx] <= Energy < E_bins_edge_HET[bin_idx+1]
bin_idx_EffArea, bin_idx_PSF = find_bin_indices(Energy)
Ebin_lb_EffArea, Ebin_ub_EffArea, Ebin_lb_PSF, Ebin_ub_PSF = find_bin_edges(Energy)
energy_exposure_map_str, energy_PSF_str = energy_str(Energy)

print(f"Energy {Energy} GeV falls in EffArea bin index: {bin_idx_EffArea} (edges: {E_bins_edge_EffArea[bin_idx_EffArea]}, {E_bins_edge_EffArea[bin_idx_EffArea+1]})")
print(f"Energy {Energy} GeV falls in PSF bin index: {bin_idx_PSF} (edges: {E_bins_edge_PSF[bin_idx_PSF]}, {E_bins_edge_PSF[bin_idx_PSF+1]})")

### Load data

In [ ]:
tree = None
f = ROOT.TFile(dataFile, "READ") 

tree = f.Get("DmlNtup")
print(tree)
print(type(tree))
rdf = ROOT.RDataFrame(tree)
df = rdf.AsNumpy(["celestialcoordBest_2", "celestialcoordBest_3", "tt_bgoTotalEcorr_GeV"])
df = pd.DataFrame(df)

df = df.rename(columns={"celestialcoordBest_2" : "lon", "celestialcoordBest_3" : "lat", 
                        "tt_bgoTotalEcorr_GeV" : "energy"})

df['lon_deg'] = df['lon']*180/np.pi
df['lat_deg'] = df['lat']*180/np.pi
df['lon'] = np.where(df['lon'] > np.pi, df['lon'] - 2*np.pi, df['lon'])

NB : we are to divide counts by exposure ; therefore we rely on the exposure map resolution for binning

### Plotting utils

In [ ]:
import matplotlib.colors as mcolors
def plot_map(lon_centers, lat_centers, map, title="", label="", output_name="", logNorm = False, figPath = None):
    """ Plot the exposure map in Mollweide projection.  
    Args:
        lon_centers (np.ndarray): 1D array of galactic longitudes in radians.
        lat_centers (np.ndarray): 1D array of galactic latitudes in radians.
        map (np.ndarray): 2D array of exposure values.
        figPath (str, optional): Path to save the figure. If None, the figure is shown.
    """

    # Convert your 1D radian lon/lat centers into 2D grid
    lon_grid, lat_grid = np.meshgrid(lon_centers, lat_centers, indexing='ij')

    # Convert to Mollweide's expected range: [-π, π] in lon, [-π/2, π/2] in lat
    lon_plot = lon_grid.T  # Transpose to match shape
    lat_plot = lat_grid.T
    map_plot = map.T  # Transpose as well to match pcolormesh layout

    # Plot
    fig = plt.figure(figsize=(10, 5))
    ax = fig.add_subplot(111, projection='mollweide')

    # Plot with pcolormesh (lon, lat must be in radians)
    if logNorm :
        norm = mcolors.LogNorm()  # Use logarithmic normalization for color scale
    else : 
        norm = None
    mesh = ax.pcolormesh(lon_plot, lat_plot, map_plot, cmap='viridis', shading='auto', norm=norm)

    # Final touches
    fig.colorbar(mesh, ax=ax, label=label)
    ax.set_title(title)
    tick_labels = np.array([150, 120, 90, 60, 30, 0, -30, -60, -90, -120, -150])
    ax.set_xticklabels(tick_labels)
    ax.xaxis.set_tick_params(labelcolor="white")
    ax.set_xlabel('Galactic Longitude')
    ax.set_ylabel('Galactic Latitude')
    ax.set_facecolor('black')
    ax.grid(True)

    if figPath :
        figName = figPath + output_name
        fig.savefig(figName)

    return fig

def plot_exposure_map(lon_centers, lat_centers, exposure_map, energy, figPath = None):
    """ Plot the exposure map in Mollweide projection.  
    Args:
        lon_centers (np.ndarray): 1D array of galactic longitudes in radians.
        lat_centers (np.ndarray): 1D array of galactic latitudes in radians.
        exposure_map (np.ndarray): 2D array of exposure values.
        figPath (str, optional): Path to save the figure. If None, the figure is shown.
    """

    title = f'Exposure Map ({energy} GeV)'
    label = 'Exposure [cm² sr s]'
    output_name = f"{energy}GeV_exposure_sum.png"

    return plot_map(lon_centers, lat_centers, exposure_map, title, label, output_name, figPath)

def plot_counts_map(lon_centers, lat_centers, counts_map, energy = None, figPath = None):

    title = "γ-rays Counts, 9 years"
    label = "Counts"
    output_name = "CountMap"
    if energy:
        title += f" | Energy : {energy} GeV"
        output_name += f"-{energy}"
    output_name += ".png"

    return plot_map(lon_centers, lat_centers, counts_map, title, label, output_name, True, figPath)

def plot_flux_map(lon_centers, lat_centers, flux_map, energy=None, figPath=None):
    title = "γ-rays Flux, 9 years"
    label = "Counts/Exposure [cm⁻2 s-1]"
    output_name = "FluxMap"
    if energy:
        title += f" | Energy : {energy} GeV"
        output_name += f"-{energy}"
    output_name += ".png"
    return plot_map(lon_centers, lat_centers, flux_map, title, label, output_name, True, figPath)


### Load exposure map

In [ ]:
# Load one exposure map for now

exposure_str, psf_str = energy_str(Energy)
exposure_map_file = np.load(resPath + "exposureMaps/" + energy_exposure_map_str + "GeV_exposure_sum.npz")
lon_edges = exposure_map_file['lon_edges']
lat_edges = exposure_map_file['lat_edges']
exposure_map = exposure_map_file['exposure_map']

lon_edges_sorted = np.sort(lon_edges)
lat_edges_sorted = np.sort(lat_edges)

lon_centers = 0.5*(lon_edges[1:] + lon_edges[:-1])
lat_centers = 0.5*(lat_edges[1:] + lat_edges[:-1])

### Plot exposure map

In [ ]:
fig = plot_exposure_map(lon_centers, lat_centers, exposure_map, Energy)

### Plot count map

In [ ]:
hist, xedges, yedges = np.histogram2d(df['lon'], df['lat'], bins=[lon_edges_sorted, lat_edges_sorted])
fig = plot_counts_map(lon_centers, lat_centers, hist, energy=energy_exposure_map_str)

### Center on target

In [ ]:
dR = 3 # degrees, radius of region of interest
lon_target = c.l.value
lat_target = c.b.value

print(f"Target coordinates: {lon_target}, {lat_target}")

lon_min = lon_target - dR
lon_max = lon_target + dR
lat_min = lat_target - dR
lat_max = lat_target + dR

dflonMin = df["lon_deg"].min()
dflonMax = df["lon_deg"].max()
dflatMin = df["lat_deg"].min()
dflatMax = df["lat_deg"].max()

print(f"Min longitude : {dflonMin}")
print(f"Max longitude : {dflonMax}")
print(f"Min latitude : {dflatMin}")
print(f"Min latitude : {dflatMax}")

print(f"Min target longitude : {lon_min}")
print(f"Max target longitude : {lon_max}")
print(f"Min target latitude : {lat_min}")
print(f"Min target latitude : {lat_max}")

In [ ]:
df_target = df.loc[np.sqrt((df["lon_deg"] - lon_target)**2 + (df["lat_deg"]-lat_target)**2) < dR]

In [ ]:
def plot_ROI(lons, lats, lon_bins=50, lat_bins=50, title = None, label = None, plot_target = True, figName = None):

    if title is None :
        title = f'2D Histogram of Entry Counts in {target} ROI'
    if label is None :
        label = 'Entry count'

    plt.figure(figsize=(8, 6))
    plt.hist2d(
        lons,
        lats,
        bins=[lon_bins, lat_bins],  # Adjust number of bins as needed
        cmap='viridis'  # Or any other colormap
    )
    plt.colorbar(label=label)

    # Plot the target coordinate as a red dot
    if plot_target:
        plt.plot(lon_target, lat_target, 'ro', markersize=8, label=f'{target} location')

    plt.xlabel('Longitude (deg)')
    plt.ylabel('Latitude (deg)')
    plt.legend()
    plt.title(title)
    if figName :
        plt.savefig(figPath + figName, dpi=400)
    plt.show()

In [ ]:
plot_ROI(df_target['lon_deg'], df_target['lat_deg'], figName=f"2D_Map_{target}.png")

### Create models